# Ejemplo de petición a AEMET con Python

Vamos a probar a obtener predicciones meteorológicas a AEMET.

Además en este ejemplo usamos la librería requests estándar de Python (que puede usarse para muchas APIs)

In [ ]:
import requests

# Necesitamos una API KEY de AEMET OpenData
# Podemos solicitar una aquí: https://opendata.aemet.es/centrodedescargas/altaUsuario? (es gratis y rápido)

# Pon tu API Key de AEMET OpenData (será una cadena muy larga de caracteres random)
API_KEY = "..."

## Lista de Municipios

Primero necestitamos el identificador del municipio que nos interese. Podemos solicitar una lista de municipios

In [ ]:

url = "https://opendata.aemet.es/opendata/api/maestro/municipios"

r = requests.get(
    url,
    headers={
        "api_key": API_KEY,
        "accept": "application/json"
    }
)
r.raise_for_status()

# URL temporal con los datos
datos_url = r.json()["datos"]

municipios = requests.get(datos_url).json()

for m in municipios[:10]:
    print(m)

{'latitud': '40º32\'54.450744"', 'id_old': '44004', 'url': 'ababuj-id44001', 'latitud_dec': '40.54845854', 'altitud': '1372', 'capital': 'Ababuj', 'num_hab': '65', 'zona_comarcal': '624401', 'destacada': '0', 'nombre': 'Ababuj', 'longitud_dec': '-0.80780117', 'id': 'id44001', 'longitud': '-0º48\'28.084212"'}
{'latitud': '40º54\'58.824504"', 'id_old': '40004', 'url': 'abades-id40001', 'latitud_dec': '40.91634014', 'altitud': '971', 'capital': 'Abades', 'num_hab': '873', 'zona_comarcal': '674001', 'destacada': '0', 'nombre': 'Abades', 'longitud_dec': '-4.26787389', 'id': 'id40001', 'longitud': '-4º16\'4.346004"'}
{'latitud': '43º8\'51.525564"', 'id_old': '48010', 'url': 'abadino-abadino-zelaieta-id48001', 'latitud_dec': '43.14764599', 'altitud': '144', 'capital': 'Abadiño-Zelaieta', 'num_hab': '7504', 'zona_comarcal': '754802', 'destacada': '0', 'nombre': 'Abadiño', 'longitud_dec': '-2.60687319', 'id': 'id48001', 'longitud': '-2º36\'24.743484"'}
{'latitud': '40º15\'34.315272"', 'id_old':

Para San Mateo de Gállego el código es 50235

## Predicción diaria

In [ ]:

# Código INE del municipio
CODIGO_MUNICIPIO = "50235"

url = (
    f"https://opendata.aemet.es/opendata/api/"
    f"prediccion/especifica/municipio/diaria/{CODIGO_MUNICIPIO}"
)

headers = {
    "accept": "application/json",
    "api_key": API_KEY
}

# Primera petición: obtiene la URL de descarga
respuesta = requests.get(url, headers=headers)
respuesta.raise_for_status()

info = respuesta.json()

if info["estado"] != 200:
    raise Exception(info["descripcion"])

# Segunda petición: descarga los datos reales
datos_url = info["datos"]

datos = requests.get(datos_url)
datos.raise_for_status()

prediccion = datos.json()

print(prediccion)

[{'origen': {'productor': 'Agencia Estatal de Meteorología - AEMET. Gobierno de España', 'web': 'https://www.aemet.es', 'enlace': 'https://www.aemet.es/es/eltiempo/prediccion/municipios/san-mateo-de-gallego-id50235', 'language': 'es', 'copyright': '© AEMET. Autorizado el uso de la información y su reproducción citando a AEMET como autora de la misma.', 'notaLegal': 'https://www.aemet.es/es/nota_legal'}, 'elaborado': '2026-07-11T11:39:10', 'nombre': 'San Mateo de GÃ¡llego', 'provincia': 'Zaragoza', 'prediccion': {'dia': [{'probPrecipitacion': [{'value': 0, 'periodo': '00-24'}, {'value': 0, 'periodo': '00-12'}, {'value': 0, 'periodo': '12-24'}, {'value': 0, 'periodo': '00-06'}, {'value': 0, 'periodo': '06-12'}, {'value': 0, 'periodo': '12-18'}, {'value': 0, 'periodo': '18-24'}], 'cotaNieveProv': [{'value': '', 'periodo': '00-24'}, {'value': '', 'periodo': '00-12'}, {'value': '', 'periodo': '12-24'}, {'value': '', 'periodo': '00-06'}, {'value': '', 'periodo': '06-12'}, {'value': '', 'peri

Ahora un poquito más estructurado

In [ ]:
url = (
    f"https://opendata.aemet.es/opendata/api/"
    f"prediccion/especifica/municipio/diaria/{CODIGO_MUNICIPIO}"
)

headers = {
    "accept": "application/json",
    "api_key": API_KEY
}

# Obtener URL temporal
r = requests.get(url, headers=headers)
r.raise_for_status()

datos_url = r.json()["datos"]

# Descargar datos
pred = requests.get(datos_url).json()

municipio = pred[0]

print("Municipio:", municipio["nombre"])
print()

for dia in municipio["prediccion"]["dia"]:
    print(f"Fecha: {dia['fecha']}")
    print("  Estado del cielo:", dia["estadoCielo"][0]["descripcion"])
    print("  Temp. máxima:", dia["temperatura"]["maxima"], "°C")
    print("  Temp. mínima:", dia["temperatura"]["minima"], "°C")
    print("  Prob. precipitación:", dia["probPrecipitacion"][0]["value"], "%")
    print()

Municipio: San Mateo de GÃ¡llego

Fecha: 2026-07-11T00:00:00
  Estado del cielo: 
  Temp. máxima: 39 °C
  Temp. mínima: 20 °C
  Prob. precipitación: 0 %

Fecha: 2026-07-12T00:00:00
  Estado del cielo: Poco nuboso
  Temp. máxima: 39 °C
  Temp. mínima: 21 °C
  Prob. precipitación: 5 %

Fecha: 2026-07-13T00:00:00
  Estado del cielo: Intervalos nubosos
  Temp. máxima: 38 °C
  Temp. mínima: 22 °C
  Prob. precipitación: 40 %

Fecha: 2026-07-14T00:00:00
  Estado del cielo: Poco nuboso
  Temp. máxima: 41 °C
  Temp. mínima: 23 °C
  Prob. precipitación: 5 %

Fecha: 2026-07-15T00:00:00
  Estado del cielo: Poco nuboso
  Temp. máxima: 41 °C
  Temp. mínima: 22 °C
  Prob. precipitación: 5 %

Fecha: 2026-07-16T00:00:00
  Estado del cielo: Intervalos nubosos
  Temp. máxima: 40 °C
  Temp. mínima: 21 °C
  Prob. precipitación: 45 %

Fecha: 2026-07-17T00:00:00
  Estado del cielo: Intervalos nubosos
  Temp. máxima: 40 °C
  Temp. mínima: 20 °C
  Prob. precipitación: 25 %



Y lo integramos en un dataframe, con el que más adelante podríamos hacer más cosas

In [ ]:
import pandas as pd

# Prepare data for DataFrame
weather_data = []
for dia in municipio["prediccion"]["dia"]:
    weather_data.append({
        "Fecha": dia["fecha"].split('T')[0], # Extract only date part
        "Estado del cielo": dia["estadoCielo"][0]["descripcion"] if dia["estadoCielo"] and dia["estadoCielo"][0]["descripcion"] else "N/A",
        "Temperatura máxima (°C)": dia["temperatura"]["maxima"],
        "Temperatura mínima (°C)": dia["temperatura"]["minima"],
        "Probabilidad de precipitación (%)": dia["probPrecipitacion"][0]["value"] if dia["probPrecipitacion"] and dia["probPrecipitacion"][0]["value"] is not None else 0
    })

# Create DataFrame
df_prediccion = pd.DataFrame(weather_data)

# Display DataFrame
display(df_prediccion)

,Fecha,Estado del cielo,Temperatura máxima (°C),Temperatura mínima (°C),Probabilidad de precipitación (%)
0,2026-07-11,N/A,39,20,0
1,2026-07-12,Poco nuboso,39,21,5
2,2026-07-13,Intervalos nubosos,38,22,40
3,2026-07-14,Poco nuboso,41,23,5
4,2026-07-15,Poco nuboso,41,22,5
5,2026-07-16,Intervalos nubosos,40,21,45
6,2026-07-17,Intervalos nubosos,40,20,25
